In [15]:
import numpy as np
from pathlib import Path
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score

In [16]:
datasets = load_dataset("csv", data_files={"train": "../data/processed/liar_binary_train.csv",
                                           "validation": "../data/processed/liar_binary_val.csv",
                                           "test": "../data/processed/liar_binary_test.csv"})

print(datasets)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 5286
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 661
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 661
    })
})


In [17]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
def tokenize(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=256)

In [18]:
tokenized_dataset = datasets.map(tokenize, batched=True)
print(tokenized_dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'input_ids', 'attention_mask'],
        num_rows: 5286
    })
    validation: Dataset({
        features: ['text', 'label', 'input_ids', 'attention_mask'],
        num_rows: 661
    })
    test: Dataset({
        features: ['text', 'label', 'input_ids', 'attention_mask'],
        num_rows: 661
    })
})


In [ ]:
tokenized_dataset = tokenized_dataset.remove_columns(["text"])
tokenized_dataset = tokenized_dataset.rename_column("label", "labels")
tokenized_dataset.set_format("torch")
print(tokenized_dataset)

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)

In [21]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    accuracy = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average="weighted")
    return {"accuracy": accuracy, "f1": f1}

In [22]:
training_args = TrainingArguments(
    output_dir = "../models/distilbert-fakenews",
    num_train_epochs = 3,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size = 32,
    learning_rate = 2e-5,
    eval_strategy = "epoch",
    save_strategy = "epoch",
    load_best_model_at_end = True,
    metric_for_best_model = "f1",
    report_to = "none"
)

In [ ]:
pip install "transformers[torch]"

In [24]:
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = tokenized_dataset["train"],
    eval_dataset = tokenized_dataset["validation"],
    compute_metrics = compute_metrics
)

In [ ]:
trainer.train()

In [26]:
trainer.save_model("../models/distilbert-fakenews")
tokenizer.save_pretrained("../models/distilbert-fakenews")

('../models/distilbert-fakenews/tokenizer_config.json',
 '../models/distilbert-fakenews/special_tokens_map.json',
 '../models/distilbert-fakenews/vocab.txt',
 '../models/distilbert-fakenews/added_tokens.json',
 '../models/distilbert-fakenews/tokenizer.json')